<a href="https://colab.research.google.com/github/anguy22/anguy22.github.io/blob/main/Assignment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Setup**

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display

# show a compact view of any result: first and last rows, plus its shape
pd.set_option("display.max_rows", 10)
pd.set_option("display.width", 120)

DB_PATH = Path("boilermaker_brews.db")

if not DB_PATH.exists():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            "Place boilermaker_brews.db in the notebook's working folder."
        ) from exc

    print("Choose boilermaker_brews.db from the course files.")
    files.upload()

if not DB_PATH.exists():
    raise FileNotFoundError("boilermaker_brews.db was not uploaded.")

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

def _execute_sql(connection, statement):
    statement = statement.strip()
    first_word = statement.split(None, 1)[0].upper()
    if first_word in {"SELECT", "WITH", "PRAGMA", "EXPLAIN"}:
        result = pd.read_sql_query(statement, connection)
        display(result)
        return

    connection.executescript(statement)
    connection.commit()
    print("Statement executed successfully.")

def _sql_magic(line, cell):
    _execute_sql(conn, cell)

def _expected_error_magic(line, cell):
    try:
        _execute_sql(conn, cell)
    except Exception as error:
        print(f"Expected error: {type(error).__name__}: {error}")
    else:
        raise AssertionError("This demonstration was expected to produce an SQL error.")

ip = get_ipython()
ip.register_magic_function(_sql_magic, "cell", "sql")
ip.register_magic_function(_expected_error_magic, "cell", "sql_expect_error")

table_count = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM sqlite_master WHERE type='table';", conn
).iloc[0, 0]
print(f"Connected to {DB_PATH.name}: {table_count} tables. The %%sql demo command is ready.")

Choose boilermaker_brews.db from the course files.


Saving boilermaker_brews.db to boilermaker_brews.db
Connected to boilermaker_brews.db: 6 tables. The %%sql demo command is ready.


**Initial Queries**

In [ ]:
%%sql
SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;


,name
0,customers
1,employees
2,order_items
3,orders
4,products
5,stores


In [ ]:
%%sql
SELECT 'stores' AS table_name, COUNT(*) AS n_rows FROM stores
UNION ALL SELECT 'products',    COUNT(*) FROM products
UNION ALL SELECT 'customers',   COUNT(*) FROM customers
UNION ALL SELECT 'employees',   COUNT(*) FROM employees
UNION ALL SELECT 'orders',      COUNT(*) FROM orders
UNION ALL SELECT 'order_items', COUNT(*) FROM order_items;


,table_name,n_rows
0,stores,6
1,products,26
2,customers,600
3,employees,36
4,orders,51927
5,order_items,78777


In [ ]:
%%sql
SELECT * FROM stores

,store_id,name,campus_area,opened_date,seats
0,1,Chauncey Hill,Chauncey,2019-08-12,38
1,2,PMU Ground Floor,Memorial Union,2017-01-09,64
2,3,Discovery Park,Discovery Park,2022-03-21,22
3,4,Levee Plaza,Levee,2020-09-01,30
4,5,State Street East,State Street,2018-05-14,26
5,6,Airport Rd Drive-Thru,South Campus,2023-10-02,0


In [ ]:
%%sql
SELECT * FROM stores s
JOIN employees e ON e.store_id = s.store_id
JOIN orders o ON o.employee_id = e.employee_id
JOIN order_items oi ON oi.order_id = o.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE s.store_id = 1

,store_id,name,campus_area,opened_date,seats,employee_id,name,store_id,hired_date,hourly_wage,...,channel,order_id,product_id,quantity,unit_price,product_id,name,category,price,is_seasonal
0,1,Chauncey Hill,Chauncey,2019-08-12,38,1,Tyler Park,1,2023-06-07,11.25,...,counter,1,10,1,5.25,10,Caramel Latte,espresso,5.25,0
1,1,Chauncey Hill,Chauncey,2019-08-12,38,2,Ethan Clark,1,2026-02-22,15.82,...,counter,2,16,1,2.75,16,Earl Grey Tea,tea,2.75,0
2,1,Chauncey Hill,Chauncey,2019-08-12,38,2,Ethan Clark,1,2026-02-22,15.82,...,counter,2,6,1,3.50,6,Americano,espresso,3.50,0
3,1,Chauncey Hill,Chauncey,2019-08-12,38,2,Ethan Clark,1,2026-02-22,15.82,...,counter,2,7,1,4.25,7,Cappuccino,espresso,4.25,0
4,1,Chauncey Hill,Chauncey,2019-08-12,38,4,Noah Shah,1,2026-03-03,14.77,...,counter,3,12,1,5.75,12,Pumpkin Spice Latte,espresso,5.75,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13430,1,Chauncey Hill,Chauncey,2019-08-12,38,4,Noah Shah,1,2026-03-03,14.77,...,app,51780,4,1,4.95,4,Nitro Cold Brew,brew,4.95,0
13431,1,Chauncey Hill,Chauncey,2019-08-12,38,4,Noah Shah,1,2026-03-03,14.77,...,app,51781,11,1,5.25,11,Mocha,espresso,5.25,0
13432,1,Chauncey Hill,Chauncey,2019-08-12,38,2,Ethan Clark,1,2026-02-22,15.82,...,counter,51782,9,1,5.00,9,Latte 16oz,espresso,5.00,0
13433,1,Chauncey Hill,Chauncey,2019-08-12,38,2,Ethan Clark,1,2026-02-22,15.82,...,counter,51782,3,1,4.25,3,Cold Brew 16oz,brew,4.25,0


# **1. Revenue and Units by Category**

The two categories which reverse their relative order are "tea" and "pastry". We can see this by viewing our focussed query in descending order, where the tea had greater revenue but pastries actually had more units sold. This explains the reversal and can be traced back to the unit_price specified in order_items, as tea brings in more revenue since its unit price is higher than that of pastry.

In [ ]:
%%sql
SELECT
    p.category,
    SUM(oi.unit_price * oi.quantity) AS revenue,
    SUM(oi.quantity) AS units_sold
FROM products p
JOIN order_items oi ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY revenue DESC

,category,revenue,units_sold
0,espresso,197025.25,41410
1,brew,67355.30,18175
2,tea,44795.50,12196
3,pastry,39935.30,13168
4,food,21793.25,2963
5,merch,3310.00,334


In [ ]:
%%sql
SELECT
    p.category,
    SUM(oi.unit_price * oi.quantity) AS revenue,
    SUM(oi.quantity) AS units_sold
FROM products p
JOIN order_items oi ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY units_sold DESC

,category,revenue,units_sold
0,espresso,197025.25,41410
1,brew,67355.30,18175
2,pastry,39935.30,13168
3,tea,44795.50,12196
4,food,21793.25,2963
5,merch,3310.00,334
